# 02 · Windows, features and participant-independent modeling

The first model compares baseline, stress-task, and amusement-task labels for participants left out of model fitting. It asks whether the recorded conditions differ in these chest signals; it does not measure a person's internal stress or establish real-world performance.

We use synchronized chest ECG, EDA and temperature at the documented 700 Hz rate. A 30-second window advances by 15 seconds, so adjacent windows overlap by 50%. Windows touching a transition or a different condition are discarded. The overlap makes random window splitting especially dangerous; all windows from each person stay in one cross-validation fold.

## 1. Confirm data and set window parameters

The feature pipeline loads one trusted participant pickle at a time and validates channel and label lengths. It will stop with a helpful message if data is missing or the actual archive differs from the expected format. If a required file is missing, the notebook stops with a path hint. It does not substitute sample data or made-up scores.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd() / "wearable-stress-ai", Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((candidate for candidate in candidates if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()), Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from src.wesad import find_subject_files
from src.features import extract_dataset_features
from src.evaluation import evaluate_grouped_cv, save_evaluation

DATA_ROOT = PROJECT_ROOT / "data" / "raw" / "WESAD"
WINDOW_SECONDS = 30
STEP_SECONDS = 15
subject_files = find_subject_files(DATA_ROOT)
DATA_AVAILABLE = bool(subject_files)
print(f"Window: {WINDOW_SECONDS}s; step: {STEP_SECONDS}s; overlap: {100*(1-STEP_SECONDS/WINDOW_SECONDS):.0f}%")
if not DATA_AVAILABLE:
    print("PENDING: WESAD data is not available locally. Follow data/README.md. Model fitting and metrics are pending.")

## 2. Extract window-level features

For each label-pure window, we calculate mean, standard deviation, minimum, maximum, median, interquartile range, root-mean-square and missing fraction for each signal. These summarize level and variability. RMS is the square root of the mean squared amplitude. They are intentionally simple features that are easy to inspect.

A strict label-pure rule avoids assigning one class to a window that straddles conditions. We target labels 1, 2 and 3 only; transition/undefined, meditation, and any other codes are excluded. Confirm numeric label meaning in the README supplied with your official release.

In [ ]:
features = None
if DATA_AVAILABLE:
    features, _ = extract_dataset_features(DATA_ROOT, WINDOW_SECONDS, STEP_SECONDS)
    display(features.head())
    print(f"Windows: {len(features):,}; participants: {features['subject'].nunique()}")
    display(features.groupby(["label", "subject"]).size().unstack(fill_value=0))
    print("Window counts by class:")
    display(features["label"].map({1:"Baseline", 2:"Stress", 3:"Amusement"}).value_counts())

## 3. Participant-independent model comparison

The simple baseline always predicts the most frequent training class. The stronger model is a class-weighted random forest, which can learn nonlinear feature interactions. Missing values are imputed using medians learned from that fold's training participants only. We use stratified grouped cross-validation: class proportions are balanced as far as possible while participants remain separate. If the data cannot support at least two participant groups per class, the pipeline reports that limitation instead of inventing a score.

In [ ]:
result = None
if DATA_AVAILABLE and features is not None:
    result = evaluate_grouped_cv(features, n_splits=5, random_state=42)
    print("Out-of-fold metrics pooled across unseen-participant folds:")
    display(result["metrics"])
    print("Per-class precision, recall and F1:")
    display(result["per_class"])
    print("Fold-level metrics:")
    display(result["fold_metrics"])
    print("Participant split audit (overlap must be zero):")
    display(result["split_audit"])
    print("Confusion matrix counts:")
    display(result["confusion"].pivot_table(index=["model", "actual"], columns="predicted", values="count", fill_value=0))
    save_evaluation(result, PROJECT_ROOT / "reports")
    print("Saved measured result tables in reports/.")
else:
    print("No model was fit. Results remain pending until official WESAD files are available.")

## 4. Interpret the evaluation carefully

Accuracy is the fraction of all predictions that are correct. Macro F1 averages class-level F1 equally, so a large class cannot dominate the score. Per-class precision answers “when predicted as this condition, how often was it correct?” Recall answers “of this condition's windows, how many were found?” Confusion matrices show which classes are mixed up.

Class imbalance matters because conditions may contribute different amounts of usable label-pure data. Participant variation matters because physiology differs across people. Grouped validation tests whether the model transfers to held-out participants in this dataset. It does not test a new device, daily life, clinical use, driving, or motorsport. Even a good score could exploit posture, task movement, recording order, or other experimental cues. The stress condition involved a speech and arithmetic task, amusement used humorous videos, and baseline involved neutral reading. The label is the experimenter's protocol condition, not a direct measurement of a person's internal state.

## Glossary and independent exercises

- **Window:** a fixed-duration segment cut from a longer recorded series.
- **Overlap:** the fraction of adjacent windows sharing samples.
- **Feature:** a compact numerical summary used by a model.
- **Macro F1:** the unweighted average of class-level F1 scores.
- **Grouped cross-validation:** folds are split by participant rather than by individual windows.
- **Leakage:** information from the test set influencing model fitting or evaluation.

Try these only after the first run is complete:

1. Compare 15-second and 60-second windows. How do window counts and feature stability change?
2. Change the step to equal the window size (no overlap). Does performance change?
3. Compare ECG-only, EDA-only and temperature-only features.
4. Add the wrist channels only after resampling or explicitly aligning their timestamps; explain the alignment choice.
5. Try leave-one-participant-out validation and report per-participant results.
6. Add signal-quality checks and compare performance before and after excluding poor-quality windows.

**Pending results:** no WESAD archive was found when this project was prepared. This notebook contains no fabricated metrics; the report will be created only by running the evaluation on obtained data.